# 5장. 분석을 믿을 수 있게 만드는 데이터 전처리

이 노트북은 `book/chapters/ch05_data_preprocessing.md` 강의안을 초보자가 그대로 따라 하며 이해할 수 있도록 구성한 실습 자료입니다.

이번 장의 핵심은 데이터를 무조건 깨끗하게 만드는 것이 아니라, 분석 결과를 신뢰할 수 있도록 데이터의 상태를 확인하고 처리 기준을 남기는 것입니다.


## 0. 이 노트북 사용 방법

아래 셀을 위에서부터 차례대로 실행하세요.

- 원본 데이터는 `data/raw/`에 그대로 둡니다.
- 전처리 결과는 `data/processed/`에 별도로 저장합니다.
- 전처리 요약 보고서는 `reports/ch05_preprocessing_summary.md`에 저장합니다.
- 코드가 실행되었다고 끝이 아니라, 처리 기준을 설명할 수 있어야 합니다.


## 1. 왜 전처리가 필요한가

현실의 데이터는 처음부터 분석하기 좋은 형태로 주어지지 않습니다.

예를 들어 다음과 같은 문제가 있을 수 있습니다.

| 문제 유형 | 예시 | 확인할 질문 |
|---|---|---|
| 결측치 | 나이가 비어 있는 고객 | 비어 있는 이유가 무엇인가? 삭제해도 되는가? |
| 중복 | 같은 고객 ID가 여러 번 등장 | 자연스러운 중복인가, 데이터 오류인가? |
| 타입 오류 | 가격이 문자열로 저장됨 | 계산 가능한 숫자형으로 바꿀 수 있는가? |
| 날짜 오류 | 주문일이 문자열로 저장됨 | 월별·요일별 분석에 사용할 수 있는가? |
| 문자열 표기 차이 | `Seoul`, ` Seoul`, `SEOUL` | 같은 값을 하나의 표기로 통일해야 하는가? |
| 이상값 | 수량이 음수이거나 가격이 0원 | 실제 의미가 있는 값인가, 입력 오류인가? |
| 파생 컬럼 필요 | 수량과 단가만 있고 주문금액이 없음 | 분석에 필요한 새 컬럼을 만들 수 있는가? |


## 2. 원본 데이터와 전처리 데이터 분리하기

전처리에서 가장 중요한 원칙은 원본 데이터를 직접 덮어쓰지 않는 것입니다.

```text
data/
├─ raw/
│  ├─ customers.csv
│  ├─ products.csv
│  ├─ orders.csv
│  └─ order_items.csv
└─ processed/
   ├─ customers_clean.csv
   ├─ products_clean.csv
   ├─ orders_clean.csv
   └─ order_items_clean.csv
```

원본과 전처리 결과를 분리하면 실수했을 때 원본으로 돌아갈 수 있고, 전처리 전후 차이를 비교할 수 있습니다.


## 3. 패키지와 경로 설정

노트북이 `notebooks/` 폴더 안에서 실행되는 경우와 프로젝트 루트에서 실행되는 경우를 모두 고려해 경로를 설정합니다.


In [235]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np

CURRENT_DIR = Path.cwd()

#if CURRENT_DIR.name == 'notebooks':
if CURRENT_DIR.name == 'chapter05':  #내 과제에서는 llm-data-analysis-course\practice\chapter05 폴더 안에서 실행
    PROJECT_ROOT = CURRENT_DIR.parent.parent # 2단계 상위 폴더로 이동하여 프로젝트 루트 경로를 설정
else:
    PROJECT_ROOT = CURRENT_DIR

#RAW_DIR = PROJECT_ROOT /'data' / 'raw'
RAW_DIR = PROJECT_ROOT / 'practice' / 'chapter05' / 'data' / 'raw'   # Chapter 05 전용 Raw 데이터
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
REPORT_DIR = PROJECT_ROOT / 'reports'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('현재 실행 위치:', CURRENT_DIR)
print('프로젝트 루트:', PROJECT_ROOT)
print('원본 데이터 폴더:', RAW_DIR)
print('전처리 데이터 폴더:', PROCESSED_DIR)
print('보고서 폴더:', REPORT_DIR)


현재 실행 위치: e:\git\llm-data-analysis-course
프로젝트 루트: e:\git\llm-data-analysis-course
원본 데이터 폴더: e:\git\llm-data-analysis-course\practice\chapter05\data\raw
전처리 데이터 폴더: e:\git\llm-data-analysis-course\data\processed
보고서 폴더: e:\git\llm-data-analysis-course\reports


## 4. 원본 데이터 불러오기

이번 장에서는 온라인 쇼핑몰 예제 데이터 4개를 사용합니다.

- `customers.csv`: 고객 정보
- `products.csv`: 상품 정보
- `orders.csv`: 주문 정보
- `order_items.csv`: 주문 상세 정보


In [236]:
customers = pd.read_csv(RAW_DIR / 'customers.csv')
products = pd.read_csv(RAW_DIR / 'products.csv')
orders = pd.read_csv(RAW_DIR / 'orders.csv')
order_items = pd.read_csv(RAW_DIR / 'order_items.csv')

print('원본 데이터 불러오기 완료')


원본 데이터 불러오기 완료


### [💫 Notes 💫] - 데스크탑 변경 예정이라 기록해둠!
---
##### 오류 해결
1. 경로가 이상할때 -> Notebook은 저장소 내부에서 실행할 것!
```python
from pathlib import Path
print(Path.cwd())
```

2. 파일이 없을 때.

```powershell
python scripts/generate_sample_data.py
```

## 5. 전처리 전 데이터 크기 기록하기

전처리 전 데이터 크기를 기록해 두면 나중에 어떤 데이터가 얼마나 바뀌었는지 비교할 수 있습니다.


In [237]:
raw_data = {
    'customers': customers,
    'products': products,
    'orders': orders,
    'order_items': order_items,
}

raw_shapes = pd.DataFrame({
    'dataset': list(raw_data.keys()),
    'rows': [df.shape[0] for df in raw_data.values()],
    'columns': [df.shape[1] for df in raw_data.values()],
})

raw_shapes


,dataset,rows,columns
0,customers,152,6
1,products,102,4
2,orders,302,5
3,order_items,765,5


In [238]:
for name, df in raw_data.items():
    print(f'[{name}]')
    print('shape:', df.shape)
    print('columns:', list(df.columns))
    print()


[customers]
shape: (152, 6)
columns: ['customer_id', 'name', 'gender', 'age', 'city', 'signup_date']

[products]
shape: (102, 4)
columns: ['product_id', 'product_name', 'category', 'price']

[orders]
shape: (302, 5)
columns: ['order_id', 'customer_id', 'order_date', 'payment_method', 'order_status']

[order_items]
shape: (765, 5)
columns: ['order_item_id', 'order_id', 'product_id', 'quantity', 'unit_price']



## 6. 원본을 복사해서 전처리 시작하기

전처리 과정에서는 원본 DataFrame을 직접 수정하지 않고 복사본을 사용합니다.


In [239]:
customers_clean = customers.copy()
products_clean = products.copy()
orders_clean = orders.copy()
order_items_clean = order_items.copy()

clean_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

print('전처리용 복사본 생성 완료')


전처리용 복사본 생성 완료


## 7. 결측치 확인하기

결측치를 처리하기 전에 먼저 어디에 얼마나 비어 있는 값이 있는지 확인합니다. 개수와 비율을 함께 보면 데이터 크기에 따른 차이를 이해하기 쉽습니다.


In [240]:
for name, df in clean_data.items():
    print(f'\n[{name}] 결측치 개수')
    print(df.isna().sum()) #True를 1, False를 0으로 계산하여 컬럼별 결측치 개수 집계



[customers] 결측치 개수
customer_id    0
name           0
gender         0
age            4
city           2
signup_date    0
dtype: int64

[products] 결측치 개수
product_id      0
product_name    0
category        0
price           0
dtype: int64

[orders] 결측치 개수
order_id          0
customer_id       0
order_date        0
payment_method    0
order_status      0
dtype: int64

[order_items] 결측치 개수
order_item_id    0
order_id         0
product_id       0
quantity         0
unit_price       0
dtype: int64


In [241]:
def missing_summary(df):
    summary = pd.DataFrame({
        'missing_count': df.isna().sum(),
        'missing_ratio': (df.isna().mean() * 100).round(2), # 각 컬럼의 결측치 비율을 소수점 둘째 자리까지 반올림하여 백분율 계산
    })
    return summary.sort_values('missing_count', ascending=False)#내림차순

missing_summary(customers_clean)


,missing_count,missing_ratio
age,4,2.63
city,2,1.32
name,0,0.00
customer_id,0,0.00
gender,0,0.00
signup_date,0,0.00


In [242]:
#변환 실패 기록 함수
def report_conversion(raw, converted, label):
    """변환 전 원본과 변환 후 값을 비교해 결측과 변환 실패를 분리해 출력"""
    raw = raw.loc[converted.index]            # 중복 제거 등으로 행이 줄었을 때 맞춰 비교
    failed = raw.notna() & converted.isna()   # 원래 값이 있었는데 변환 후 비어버린 것
    print(f'[{label}] 원래 결측: {raw.isna().sum()}건 / 변환 실패: {failed.sum()}건')
    if failed.any():
        print('   실패 값:', raw[failed].tolist())
    return failed

## 8. 결측치 처리하기

결측치 처리는 컬럼의 의미에 따라 달라집니다.

- 나이처럼 숫자형 컬럼은 중앙값으로 대체할 수 있습니다.
- 도시처럼 범주형 컬럼은 `Unknown`으로 남겨 분석에서 구분할 수 있습니다.

실제 업무에서는 결측치가 왜 생겼는지 먼저 확인해야 합니다.


In [243]:
if 'age' in customers_clean.columns: #age 컬럼이 존재하는지 확인
    customers_clean['age'] = pd.to_numeric(customers_clean['age'], errors='coerce')  # age 컬럼을 숫자형 데이터로 변환, 숫자로 변환할 수 없는 값은 NaN으로 처리
    report_conversion(customers['age'], customers_clean['age'], 'age')     # 변환하는 셀마다 기록 한 줄씩 추가
    print('범위 밖 나이(18 미만, 100 초과):',
          customers_clean.loc[(customers_clean['age'] < 18) | (customers_clean['age'] > 100), ['customer_id', 'age']].values.tolist())  # 변환하는 셀마다 기록 한 줄씩 추가
    
    age_median = customers_clean['age'].median() # age 컬럼에서 결측치를 제외한 값들의 중앙값 계산
    customers_clean['age'] = customers_clean['age'].fillna(age_median) # fillna(): 결측치를 지정한 값으로 채우는 함수
    print('age 중앙값:', age_median)

if 'city' in customers_clean.columns:
    customers_clean['city'] = customers_clean['city'].fillna('Unknown')  # city 컬럼의 결측치를 'Unknown' 문자열로 대체

missing_summary(customers_clean)


[age] 원래 결측: 4건 / 변환 실패: 2건
   실패 값: ['unknown', 'unknown']
범위 밖 나이(18 미만, 100 초과): [[33.0, 150.0]]
age 중앙값: 41.0


,missing_count,missing_ratio
customer_id,0,0.0
name,0,0.0
gender,0,0.0
age,0,0.0
city,0,0.0
signup_date,0,0.0


## 9. 중복 확인하기

중복은 맥락에 따라 의미가 다릅니다. `customers`의 `customer_id` 중복은 오류일 가능성이 크지만, `order_items`의 `order_id` 반복은 한 주문에 여러 상품이 들어갈 수 있으므로 자연스럽습니다.


In [244]:
for name, df in clean_data.items():
    print(name, '전체 행 중복 수:', df.duplicated().sum()) #True의 개수를 합산하여 중복 행 개수 계산


customers 전체 행 중복 수: 2
products 전체 행 중복 수: 2
orders 전체 행 중복 수: 2
order_items 전체 행 중복 수: 2


In [245]:
key_checks = {
    'customers': ('customer_id', customers_clean),
    'products': ('product_id', products_clean),
    'orders': ('order_id', orders_clean),
    'order_items': ('order_item_id', order_items_clean),
}

for name, (key_col, df) in key_checks.items():
    if key_col in df.columns:
        print(name, key_col, '중복 수:', df[key_col].duplicated().sum())
    else:
        print(name, key_col, '컬럼 없음')


customers customer_id 중복 수: 2
products product_id 중복 수: 2
orders order_id 중복 수: 2
order_items order_item_id 중복 수: 2


In [246]:
customers_clean = customers_clean.drop_duplicates()
products_clean = products_clean.drop_duplicates()
orders_clean = orders_clean.drop_duplicates()
order_items_clean = order_items_clean.drop_duplicates()

print('완전 중복 행 제거 완료')


완전 중복 행 제거 완료


## 10. 문자열 표기 정리하기

문자열 데이터에는 앞뒤 공백이 숨어 있을 수 있습니다. `Seoul`과 ` Seoul `은 눈으로는 비슷하지만 pandas에서는 서로 다른 값입니다.

결측치를 문자열 `nan`으로 바꾸지 않도록 주의하면서 문자열 컬럼의 앞뒤 공백을 제거합니다.


### [💫 Notes 💫]

- isna(): 현재 값이 결측치인지 확인
- astype(str): 값을 문자열로 변환
- str.strip(): 문자열의 앞뒤 공백 및 줄바꿈 등을 제거
- where()의 조건이 False인 위치에 해당 결과를 사용
- strip_string_columns 문자열 컬럼에서 앞뒤 공백 제거

In [247]:
def strip_string_columns(df):
    result = df.copy()
    string_columns = result.select_dtypes(include='object').columns # 데이터 타입이 object인 컬럼들의 이름을 가져옴 (일반적으로 문자열 컬럼이 포함)

    for col in string_columns:
        result[col] = result[col].where(
            result[col].isna(),# True이면 원래 결측치를 그대로 유지
            result[col].astype(str).str.strip()
        ) #where(조건, 대체값)은 조건이 참이면 기존 값을 유지하고, 거짓이면 대체값을 사용

    return result

customers_clean = strip_string_columns(customers_clean) 
products_clean = strip_string_columns(products_clean)
orders_clean = strip_string_columns(orders_clean)
order_items_clean = strip_string_columns(order_items_clean)

print('문자열 앞뒤 공백 제거 완료')


문자열 앞뒤 공백 제거 완료


C:\Users\apdlh\AppData\Local\Temp\ipykernel_24812\3581284410.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = result.select_dtypes(include='object').columns # 데이터 타입이 object인 컬럼들의 이름을 가져옴 (일반적으로 문자열 컬럼이 포함)
C:\Users\apdlh\AppData\Local\Temp\ipykernel_24812\3581284410.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this wa

In [248]:
if 'city' in customers_clean.columns:
    display(customers_clean['city'].value_counts().head(10))

if 'order_status' in orders_clean.columns:
    display(orders_clean['order_status'].value_counts())


city
부산    23
대구    21
광주    21
서울    18
울산    17
인천    16
수원    10
성남     9
고양     8
대전     5
Name: count, dtype: int64

order_status
completed    175
cancelled     58
refunded      52
Complete       2
cancel         2
Refund         2
complete       1
COMPLETED      1
완료             1
Cancel         1
CANCELLED      1
취소             1
refund         1
REFUNDED       1
환불             1
Name: count, dtype: int64

In [249]:
print('표준화 전:', products_clean['category'].value_counts().to_dict())
products_clean['category'] = products_clean['category'].str.replace(' ', '', regex=False)
print('표준화 후:', products_clean['category'].value_counts().to_dict())

표준화 전: {'패션': 17, '뷰티': 17, '식품': 16, '도서': 14, '전자기기': 13, '스포츠': 11, '생활용품': 9, '전자 기기': 1, '생활 용품': 1, '뷰 티': 1}
표준화 후: {'뷰티': 18, '패션': 17, '식품': 16, '도서': 14, '전자기기': 14, '스포츠': 11, '생활용품': 10}


## 11. 주문 상태값 통일하기

주문 상태값이 `complete`, `Complete`, `COMPLETED`, `완료`처럼 섞여 있으면 같은 상태를 서로 다른 값으로 집계할 수 있습니다. 대표 표기로 통일합니다.


In [250]:
status_map = {
    'complete': 'completed',
    'Complete': 'completed',
    'COMPLETED': 'completed',
    '완료': 'completed',
    'cancel': 'cancelled',
    'Cancel': 'cancelled',
    'CANCELLED': 'cancelled',
    '취소': 'cancelled',
    'refund': 'refunded',
    'Refund': 'refunded',
    'REFUNDED': 'refunded',
    '환불': 'refunded',
}

if 'order_status' in orders_clean.columns:
    orders_clean['order_status'] = orders_clean['order_status'].replace(status_map)
    display(orders_clean['order_status'].value_counts())


order_status
completed    180
cancelled     63
refunded      57
Name: count, dtype: int64

## 12. 날짜 컬럼 변환하기

날짜처럼 보이는 값도 실제로는 문자열일 수 있습니다. 월별 주문 분석이나 요일별 분석을 하려면 날짜형으로 변환해야 합니다.

`errors="coerce"`는 변환할 수 없는 값을 `NaT`로 바꿉니다. 그래서 변환 후 실패 건수를 반드시 확인해야 합니다.


In [251]:
orders_clean['order_date'] = pd.to_datetime(orders_clean['order_date'], errors='coerce')
report_conversion(orders['order_date'], orders_clean['order_date'], 'order_date')

print('order_date 변환 실패:', orders_clean['order_date'].isna().sum())
print('주문 시작일:', orders_clean['order_date'].min())
print('주문 종료일:', orders_clean['order_date'].max())

orders_clean['order_month'] = orders_clean['order_date'].dt.to_period('M').astype(str)
orders_clean['order_dayofweek'] = orders_clean['order_date'].dt.day_name()

orders_clean[['order_date', 'order_month', 'order_dayofweek']].head()


[order_date] 원래 결측: 0건 / 변환 실패: 3건
   실패 값: ['not-a-date', '2026-02-30', '2026/99/99']
order_date 변환 실패: 3
주문 시작일: 2025-07-01 00:00:00
주문 종료일: 2028-12-31 00:00:00


,order_date,order_month,order_dayofweek
0,2026-07-12,2026-07,Sunday
1,2025-10-02,2025-10,Thursday
2,2026-03-13,2026-03,Friday
3,2025-10-12,2025-10,Sunday
4,2026-04-17,2026-04,Friday


In [252]:
if 'signup_date' in customers_clean.columns:
    customers_clean['signup_date'] = pd.to_datetime(customers_clean['signup_date'], errors='coerce')
    
    report_conversion(customers['signup_date'], customers_clean['signup_date'], 'signup_date')
    print('signup_date 변환 실패:', customers_clean['signup_date'].isna().sum())
    display(customers_clean[['customer_id', 'signup_date']].head())


[signup_date] 원래 결측: 0건 / 변환 실패: 2건
   실패 값: ['2026-13-40', 'not-a-date']
signup_date 변환 실패: 2


,customer_id,signup_date
0,1,2025-10-05
1,2,2023-09-20
2,3,2025-01-22
3,4,2025-03-13
4,5,2024-02-10


## 13. 숫자형 컬럼 변환하기

가격, 수량, 단가는 숫자처럼 보여도 문자열일 수 있습니다. 특히 `10,000`처럼 쉼표가 들어간 값은 바로 계산하기 어렵습니다.


In [253]:
def to_number(series):
    return pd.to_numeric(series.astype(str).str.replace(',', '', regex=False),errors='coerce')

products_clean['price'] = to_number(products_clean['price'])
order_items_clean['quantity'] = to_number(order_items_clean['quantity'])
order_items_clean['unit_price'] = to_number(order_items_clean['unit_price'])

report_conversion(products['price'], products_clean['price'], 'price')
report_conversion(order_items['quantity'], order_items_clean['quantity'], 'quantity')
report_conversion(order_items['unit_price'], order_items_clean['unit_price'], 'unit_price')

print('price 변환 실패:', products_clean['price'].isna().sum())
print('quantity 변환 실패:', order_items_clean['quantity'].isna().sum())
print('unit_price 변환 실패:', order_items_clean['unit_price'].isna().sum())


[price] 원래 결측: 0건 / 변환 실패: 2건
   실패 값: ['unknown', 'unknown']
[quantity] 원래 결측: 0건 / 변환 실패: 5건
   실패 값: ['3개', '2개', '5개', 'three', 'unknown']
[unit_price] 원래 결측: 0건 / 변환 실패: 2건
   실패 값: ['unknown', 'not-a-number']
price 변환 실패: 2
quantity 변환 실패: 5
unit_price 변환 실패: 2


## 14. 이상값 후보 확인하기

이상값은 일반적인 범위를 벗어난 값입니다. 하지만 이상값이 항상 오류는 아닙니다. 0원 상품은 이벤트 상품일 수 있고, 음수 수량은 반품을 의미할 수도 있습니다. 먼저 확인하고 기준을 정해야 합니다.


In [254]:
display(products_clean['price'].describe())
display(order_items_clean[['quantity', 'unit_price']].describe())


count    9.800000e+01
mean     2.005102e+05
std      1.001829e+06
min     -2.000000e+04
25%      5.200000e+04
50%      9.250000e+04
75%      1.527500e+05
max      9.999999e+06
Name: price, dtype: float64

,quantity,unit_price
count,758.000000,761.000000
mean,3.168865,103494.086728
std,4.159055,56969.823769
min,-1.000000,0.000000
25%,2.000000,55000.000000
50%,3.000000,96000.000000
75%,4.000000,154000.000000
max,100.000000,197000.000000


In [255]:
print('price <= 0:', len(products_clean[products_clean['price'] <= 0]))
print('quantity <= 0:', len(order_items_clean[order_items_clean['quantity'] <= 0]))
print('unit_price <= 0:', len(order_items_clean[order_items_clean['unit_price'] <= 0]))


price <= 0: 2
quantity <= 0: 3
unit_price <= 0: 1


### 실습용 이상값 처리 기준

이번 실습에서는 정상 주문 분석을 위해 다음 기준을 적용합니다.

```text
- price가 0 이하인 상품은 분석 대상에서 제외
- quantity가 0 이하인 주문 상세는 분석 대상에서 제외
- unit_price가 0 이하인 주문 상세는 분석 대상에서 제외
```

실제 업무에서는 이 기준을 적용하기 전에 반드시 원본 시스템이나 담당자 확인이 필요합니다.


In [256]:
#> 0 필터로 빠지는 행 나눠 보기
def filter_breakdown(s, label):
    na, le0 = s.isna().sum(), (s <= 0).sum()
    print(f'[{label}] 변환 실패(NaN): {na}건 / 0 이하: {le0}건 → 제외 대상 {(s.isna() | (s <= 0)).sum()}건')

filter_breakdown(products_clean['price'], 'price')
filter_breakdown(order_items_clean['quantity'], 'quantity')
filter_breakdown(order_items_clean['unit_price'], 'unit_price')

excluded_items = order_items_clean[
    order_items_clean['quantity'].isna() | (order_items_clean['quantity'] <= 0) |
    order_items_clean['unit_price'].isna() | (order_items_clean['unit_price'] <= 0)
]
display(excluded_items)

[price] 변환 실패(NaN): 2건 / 0 이하: 2건 → 제외 대상 4건
[quantity] 변환 실패(NaN): 5건 / 0 이하: 3건 → 제외 대상 8건
[unit_price] 변환 실패(NaN): 2건 / 0 이하: 1건 → 제외 대상 3건


,order_item_id,order_id,product_id,quantity,unit_price
16,17,8,76,NaN,56000.0
90,91,42,65,NaN,193000.0
204,205,84,41,NaN,119000.0
276,277,111,30,5.0,NaN
318,319,129,57,NaN,96000.0
432,433,171,87,5.0,NaN
446,447,175,79,NaN,153000.0
510,511,201,17,0.0,76000.0
578,579,229,17,2.0,0.0
611,612,241,24,-1.0,42000.0


In [257]:
#이상값 후보 목록
def iqr_outliers(s):
    q1, q3 = s.quantile([0.25, 0.75])
    upper, lower = q3 + 1.5 * (q3 - q1), q1 - 1.5 * (q3 - q1)
    return s[(s > upper) | (s < lower)], lower, upper

rows = []
for label, s in [('price', products_clean['price']),
                 ('quantity', order_items_clean['quantity']),
                 ('unit_price', order_items_clean['unit_price'])]:
    out, lo, up = iqr_outliers(s)
    print(f'{label}: IQR 기준 정상 범위 {lo:,.0f} ~ {up:,.0f} / 후보 {len(out)}건 → {out.tolist()}')

display(customers_clean[(customers_clean['age'] < 18) | (customers_clean['age'] > 100)])

price: IQR 기준 정상 범위 -99,125 ~ 303,875 / 후보 1건 → [9999999.0]
quantity: IQR 기준 정상 범위 -1 ~ 7 / 후보 2건 → [50.0, 100.0]
unit_price: IQR 기준 정상 범위 -93,500 ~ 302,500 / 후보 0건 → []


,customer_id,name,gender,age,city,signup_date
32,33,권승윤,F,150.0,고양,2025-09-14


In [258]:
products_clean = products_clean[products_clean['price'] > 0]
order_items_clean = order_items_clean[order_items_clean['quantity'] > 0]
order_items_clean = order_items_clean[order_items_clean['unit_price'] > 0]

print('이상값 처리 후 products:', products_clean.shape)
print('이상값 처리 후 order_items:', order_items_clean.shape)


이상값 처리 후 products: (96, 4)
이상값 처리 후 order_items: (752, 5)


## 15. 파생 컬럼 만들기

전처리된 수량과 단가를 사용하면 주문 상세 금액을 계산할 수 있습니다.

`line_total = quantity × unit_price`


In [259]:
order_items_clean['line_total'] = (
    order_items_clean['quantity'] * order_items_clean['unit_price']
)

order_items_clean[['quantity', 'unit_price', 'line_total']].head()


,quantity,unit_price,line_total
0,2.0,172000.0,344000.0
1,4.0,79000.0,316000.0
2,3.0,87000.0,261000.0
3,1.0,187000.0,187000.0
4,5.0,125000.0,625000.0


In [260]:
print('전처리 후 주문 상세 금액 합계:', order_items_clean['line_total'].sum())


전처리 후 주문 상세 금액 합계: 252140000.0


## 16. 파일 간 관계 다시 확인하기

전처리 과정에서 일부 행을 삭제하면 파일 간 관계가 깨질 수 있습니다. 예를 들어 `products`에서 일부 상품을 제외했는데 `order_items`에는 그 상품 ID가 남아 있을 수 있습니다.


In [261]:
invalid_customers = orders_clean[
    ~orders_clean['customer_id'].isin(customers_clean['customer_id'])
]

invalid_orders = order_items_clean[
    ~order_items_clean['order_id'].isin(orders_clean['order_id'])
]

invalid_products = order_items_clean[
    ~order_items_clean['product_id'].isin(products_clean['product_id'])
]

relationship_checks = pd.DataFrame({
    'check': [
        'orders.customer_id exists in customers.customer_id',
        'order_items.order_id exists in orders.order_id',
        'order_items.product_id exists in products.product_id',
    ],
    'invalid_count': [
        len(invalid_customers),
        len(invalid_orders),
        len(invalid_products),
    ],
})

relationship_checks


,check,invalid_count
0,orders.customer_id exists in customers.custome...,2
1,order_items.order_id exists in orders.order_id,2
2,order_items.product_id exists in products.prod...,26


In [262]:
#FK 불일치의 원인 나누기
removed_products = set(products['product_id']) - set(products_clean['product_id'])
invalid_products = invalid_products.copy()
invalid_products['원인'] = np.where(
    invalid_products['product_id'].isin(removed_products),
    '전처리에서 제외된 상품을 참조', '원본 products에 없는 상품')

display(invalid_products['원인'].value_counts())
display(invalid_customers[['order_id', 'customer_id']])
display(invalid_orders[['order_item_id', 'order_id']])

원인
전처리에서 제외된 상품을 참조      24
원본 products에 없는 상품     2
Name: count, dtype: int64

,order_id,customer_id
125,126,9991
277,278,9992


,order_item_id,order_id
149,150,9991
619,620,9992


In [263]:
# FK 불일치 행을 분석 대상에서 제외
before_orders, before_items = len(orders_clean), len(order_items_clean)

orders_clean = orders_clean[orders_clean['customer_id'].isin(customers_clean['customer_id'])]
order_items_clean = order_items_clean[
    order_items_clean['order_id'].isin(orders_clean['order_id']) &
    order_items_clean['product_id'].isin(products_clean['product_id'])
]

print('FK 불일치로 제외된 orders:', before_orders - len(orders_clean), '행')
print('FK 불일치로 제외된 order_items:', before_items - len(order_items_clean), '행')
print('제외 후 orders.customer_id 불일치:', (~orders_clean['customer_id'].isin(customers_clean['customer_id'])).sum())
print('제외 후 order_items.order_id 불일치:', (~order_items_clean['order_id'].isin(orders_clean['order_id'])).sum())
print('제외 후 order_items.product_id 불일치:', (~order_items_clean['product_id'].isin(products_clean['product_id'])).sum())

FK 불일치로 제외된 orders: 2 행
FK 불일치로 제외된 order_items: 31 행
제외 후 orders.customer_id 불일치: 0
제외 후 order_items.order_id 불일치: 0
제외 후 order_items.product_id 불일치: 0


## 17. 전처리 전후 비교하기

전처리 전후 데이터 크기를 비교합니다. 행 수가 줄었다면 어떤 기준으로 줄었는지 설명할 수 있어야 하고, 열 수가 늘었다면 어떤 파생 컬럼이 추가되었는지 설명할 수 있어야 합니다.


In [264]:
processed_data = {
    'customers': customers_clean,
    'products': products_clean,
    'orders': orders_clean,
    'order_items': order_items_clean,
}

processed_shapes = pd.DataFrame({
    'dataset': list(processed_data.keys()),
    'rows': [df.shape[0] for df in processed_data.values()],
    'columns': [df.shape[1] for df in processed_data.values()],
})

comparison = raw_shapes.merge(
    processed_shapes,
    on='dataset',
    suffixes=('_raw', '_processed')
)

comparison


,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,152,6,150,6
1,products,102,4,96,4
2,orders,302,5,298,7
3,order_items,765,5,721,6


## 18. 전처리 결과 저장하기

전처리된 데이터는 `data/processed` 폴더에 저장합니다. Excel에서 한글 CSV를 바로 열 수 있도록 `utf-8-sig` 인코딩을 사용합니다.


In [265]:
customers_clean.to_csv(PROCESSED_DIR / 'customers_clean.csv', index=False, encoding='utf-8-sig')
products_clean.to_csv(PROCESSED_DIR / 'products_clean.csv', index=False, encoding='utf-8-sig')
orders_clean.to_csv(PROCESSED_DIR / 'orders_clean.csv', index=False, encoding='utf-8-sig')
order_items_clean.to_csv(PROCESSED_DIR / 'order_items_clean.csv', index=False, encoding='utf-8-sig')

list(PROCESSED_DIR.glob('*_clean.csv'))


[WindowsPath('e:/git/llm-data-analysis-course/data/processed/customers_clean.csv'),
 WindowsPath('e:/git/llm-data-analysis-course/data/processed/orders_clean.csv'),
 WindowsPath('e:/git/llm-data-analysis-course/data/processed/order_items_clean.csv'),
 WindowsPath('e:/git/llm-data-analysis-course/data/processed/products_clean.csv')]

## 19. 전처리 요약 보고서 저장하기

전처리 기준과 결과를 Markdown 파일로 남겨 두면 이후 보고서 작성이 쉬워집니다.


In [266]:
summary_text = f'''# Chapter 5 데이터 전처리 요약

## 전처리 결과 파일

- customers_clean.csv
- products_clean.csv
- orders_clean.csv
- order_items_clean.csv

## 전처리 전후 데이터 크기

```text
{comparison.to_string(index=False)}
```

## 파일 간 관계 점검 결과

```text
{relationship_checks.to_string(index=False)}
```

## 주요 처리 내용

- 원본 데이터를 직접 수정하지 않고 복사본을 사용함
- 문자열 컬럼 앞뒤 공백 제거
- 고객 나이 결측치는 중앙값으로 대체
- 고객 도시 결측치는 Unknown으로 처리
- 주문 상태값 표기 통일
- 날짜 컬럼 변환 및 주문 월/요일 파생 컬럼 생성
- 가격, 수량, 단가를 숫자형으로 변환
- 0 이하 가격, 수량, 단가 확인 및 처리
- line_total 파생 컬럼 생성
- 파일 간 키 관계 재확인

## 주의 사항

이 전처리 기준은 실습용 예시입니다. 실제 업무에서는 결측치와 이상값을 삭제하거나 대체하기 전에 원본 시스템, 수집 과정, 업무 담당자 확인이 필요합니다.
'''

report_path = REPORT_DIR / 'ch05_preprocessing_summary.md'
report_path.write_text(summary_text, encoding='utf-8')

print('요약 보고서 저장 완료:', report_path)


요약 보고서 저장 완료: e:\git\llm-data-analysis-course\reports\ch05_preprocessing_summary.md


## 20. 전처리 과정을 함수로 정리하기

전처리 코드를 한 번만 실행하고 끝내면 재사용하기 어렵습니다. 같은 데이터를 다시 받거나 처리 기준을 조금 바꾸어야 할 때는 함수로 정리된 코드가 훨씬 유용합니다.


In [267]:
def preprocess_customers(df):
    result = strip_string_columns(df)

    if 'age' in result.columns:
        # to_numeric(): 숫자형 변환, errors='coerce': 변환 실패 시 NaN
        result['age'] = pd.to_numeric(result['age'], errors='coerce')
        # fillna(): 결측치 대체, median(): 중앙값 계산
        result['age'] = result['age'].fillna(result['age'].median())

    if 'city' in result.columns:
        # fillna(): 결측치를 'Unknown'으로 대체
        result['city'] = result['city'].fillna('Unknown')

    if 'signup_date' in result.columns:
        # to_datetime(): 날짜형 변환, 변환 실패 시 NaT
        result['signup_date'] = pd.to_datetime(result['signup_date'], errors='coerce')

    # drop_duplicates(): 전체 행 중복 제거
    return result.drop_duplicates()


def preprocess_products(df):
    result = strip_string_columns(df)

    if 'price' in result.columns:
        result['price'] = to_number(result['price'])
        result = result[result['price'] > 0]

    # drop_duplicates(): 전체 행 중복 제거
    return result.drop_duplicates()


def preprocess_orders(df):
    result = strip_string_columns(df)

    if 'order_status' in result.columns:
        # replace(): 지정한 값으로 치환
        result['order_status'] = result['order_status'].replace(status_map)

    if 'order_date' in result.columns:
        # to_datetime(): 날짜형 변환
        result['order_date'] = pd.to_datetime(result['order_date'], errors='coerce')
        # dt.to_period('M'): 월 단위 변환, astype(str): 문자열 변환
        result['order_month'] = result['order_date'].dt.to_period('M').astype(str)
        # dt.day_name(): 날짜에서 요일 이름 추출
        result['order_dayofweek'] = result['order_date'].dt.day_name()

    # drop_duplicates(): 전체 행 중복 제거
    return result.drop_duplicates()


def preprocess_order_items(df):
    result = strip_string_columns(df)

    if 'quantity' in result.columns:
        result['quantity'] = to_number(result['quantity'])
        result = result[result['quantity'] > 0]

    if 'unit_price' in result.columns:
        result['unit_price'] = to_number(result['unit_price'])
        result = result[result['unit_price'] > 0]

    # issubset(): 지정한 컬럼이 모두 존재하는지 확인
    if {'quantity', 'unit_price'}.issubset(result.columns):
        result['line_total'] = result['quantity'] * result['unit_price']

    # drop_duplicates(): 전체 행 중복 제거
    return result.drop_duplicates()

In [268]:
customers_clean_func = preprocess_customers(customers)
products_clean_func = preprocess_products(products)
orders_clean_func = preprocess_orders(orders)
order_items_clean_func = preprocess_order_items(order_items)

print(customers_clean_func.shape)
print(products_clean_func.shape)
print(orders_clean_func.shape)
print(order_items_clean_func.shape)


(150, 6)
(96, 4)
(300, 7)
(752, 6)


C:\Users\apdlh\AppData\Local\Temp\ipykernel_24812\3581284410.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = result.select_dtypes(include='object').columns # 데이터 타입이 object인 컬럼들의 이름을 가져옴 (일반적으로 문자열 컬럼이 포함)
C:\Users\apdlh\AppData\Local\Temp\ipykernel_24812\3581284410.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this wa

## 21. 소스 모듈 사용하기

위에서 직접 작성한 전처리 함수는 `src/preprocessing.py`에도 정리되어 있습니다. 반복 작업이나 실제 프로젝트에서는 노트북 안에만 코드를 두기보다 소스 모듈로 분리하는 것이 좋습니다.


In [269]:
from src.data_loader import load_sales_data
from src.preprocessing import (
    build_preprocessing_report,
    compare_shapes,
    duplicate_summary,
    preprocess_sales_data,
    save_processed_data,
    validate_relationships,
)

module_raw_data = load_sales_data(RAW_DIR)
module_processed_data = preprocess_sales_data(module_raw_data)
module_relationship_checks = validate_relationships(module_processed_data)
module_comparison = compare_shapes(module_raw_data, module_processed_data)

module_comparison


,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,152,6,150,6
1,order_items,765,5,752,6
2,orders,302,5,300,7
3,products,102,4,96,4


In [270]:
duplicate_summary(
    module_processed_data,
    key_columns={
        'customers': 'customer_id',
        'products': 'product_id',
        'orders': 'order_id',
        'order_items': 'order_item_id',
    },
)


,dataset,row_duplicate_count,key_column,key_duplicate_count
0,customers,0,customer_id,0
1,products,0,product_id,0
2,orders,0,order_id,0
3,order_items,0,order_item_id,0


In [271]:
module_relationship_checks


,check,invalid_count
0,orders.customer_id exists in customers.custome...,2
1,order_items.order_id exists in orders.order_id,2
2,order_items.product_id exists in products.prod...,26


## 22. 스크립트로 한 번에 실행하기

노트북에서 한 단계씩 이해한 전처리 과정을 스크립트로도 실행할 수 있습니다. 터미널에서 프로젝트 루트 기준으로 아래 명령을 실행합니다.

```bash
python scripts/preprocess_data.py
```

이 스크립트는 전처리 결과 CSV와 요약 보고서를 자동으로 저장합니다.


## 23. LLM과 함께 전처리 코드를 검토하기

LLM에게 전처리 코드를 요청할 때는 실제 고객명, 이메일, 주문 내역을 그대로 입력하지 않는 것이 좋습니다. 컬럼명, 데이터 타입, 결측치 개수, 중복 여부, 처리 목적처럼 구조화된 정보만 제공하세요.

```text
다음 전처리 코드가 안전한지 검토해 주세요.

customers["age"] = customers["age"].fillna(customers["age"].mean())
customers = customers.drop_duplicates()
orders["order_date"] = pd.to_datetime(orders["order_date"])
order_items["line_total"] = order_items["quantity"] * order_items["unit_price"]

검토 기준:
- 원본 데이터를 직접 수정하는 문제가 있는지
- 결측치 처리 방식이 적절한지
- 날짜 변환 실패를 확인하는지
- 중복 제거 기준이 충분한지
- line_total 계산 전에 숫자형 변환이 필요한지
- 더 안전한 코드로 어떻게 수정할 수 있는지
```


## 24. 실습 과제

아래 과제를 직접 해결해 보세요.

1. 각 데이터셋의 결측치 비율을 하나의 표로 합쳐 보세요.
2. `customers_clean`에서 나이가 18세 미만이거나 100세 초과인 값이 있는지 확인하세요.
3. `orders_clean`에서 월별 주문 건수를 계산하세요.
4. `order_items_clean`에서 `line_total`이 큰 순서대로 상위 10개를 확인하세요.
5. 전처리 전후 행 수가 줄어든 데이터셋이 있는지 설명해 보세요.
6. LLM에게 결측치 처리 기준을 검토해 달라는 프롬프트를 직접 작성해 보세요.


In [272]:
# 과제 1. 각 데이터셋의 결측치 비율을 하나의 표로 합쳐 보세요.
# hint: missing_summary(df).reset_index()를 활용해 보세요.

missing_tables = []
for name, df in raw_data.items():
    summary = missing_summary(df).reset_index()
    summary = summary.rename(columns={'index': 'column'})
    summary.insert(0, 'dataset', name)
    missing_tables.append(summary)

missing_table = pd.concat(missing_tables, ignore_index=True)

display(missing_table)

,dataset,column,missing_count,missing_ratio
0,customers,age,4,2.63
1,customers,city,2,1.32
2,customers,name,0,0.00
3,customers,customer_id,0,0.00
4,customers,gender,0,0.00
5,customers,signup_date,0,0.00
6,products,product_id,0,0.00
7,products,product_name,0,0.00
8,products,category,0,0.00
9,products,price,0,0.00


과제 1 결과 해석: 원본 customers의 age에서 4건(2.63%), city에서 2건(1.32%)의 결측이 있었고, 나머지 데이터셋에는 결측이 없었다. 다만 타입 변환 과정에서 age 2건, 날짜 5건의 결측이 새로 생기므로 전처리 후에도 결측을 다시 확인해야 한다.

In [273]:
# 과제 2. customers_clean에서 나이가 18세 미만이거나 100세 초과인 값이 있는지 확인하세요.

invalid_age = customers_clean[(customers_clean['age'] < 18) |(customers_clean['age'] > 100)]

print('범위 밖 나이 개수:', len(invalid_age))
print('최소 나이:', customers_clean['age'].min())
print('최대 나이:', customers_clean['age'].max())

display(invalid_age)

범위 밖 나이 개수: 1
최소 나이: 18.0
최대 나이: 150.0


,customer_id,name,gender,age,city,signup_date
32,33,권승윤,F,150.0,고양,2025-09-14


과제 2 결과 해석: 범위를 벗어난 나이는 1건(customer_id 33, 150세)이고, 최소 18세, 최대 150세이다. 150세는 입력 오류일 가능성이 높지만 원본 확인 전까지는 삭제하지 않고 이상값 후보로 표시했다. 18~100세는 이번 검증에서 정한 기준일 뿐 절대적인 유효성 기준은 아니다.

In [274]:
# 과제 3. orders_clean에서 월별 주문 건수를 계산하세요.

monthly_orders = (orders_clean.groupby('order_month').size().reset_index(name='order_count').sort_values('order_month'))

display(monthly_orders)

,order_month,order_count
0,2025-07,19
1,2025-08,26
2,2025-09,12
3,2025-10,22
4,2025-11,23
5,2025-12,23
6,2026-01,25
7,2026-02,19
8,2026-03,22
9,2026-04,26


과제 3 결과 해석: 월별 주문 건수는 2025-08과 2026-04가 26건으로 가장 많고, 정상 기간 중에서는 2026-07이 11건으로 가장 적다. 2027-01과 2028-12는 각 1건으로, 미래 날짜 주문이라 이상값 후보로 봐야 한다. 주문일 변환에 실패한 3건은 월을 알 수 없어 집계에서 빠졌다(월별 합계 295건 + 3건 = 298건). 이 결과는 모든 주문 상태를 포함한다.

In [275]:
# 과제 4. order_items_clean에서 line_total이 큰 순서대로 상위 10개를 확인하세요.
top10_items = (order_items_clean.sort_values('line_total', ascending=False).head(10))

display(top10_items)

,order_item_id,order_id,product_id,quantity,unit_price,line_total
750,751,295,31,100.0,155000.0,15500000.0
737,738,290,96,50.0,96000.0,4800000.0
319,320,129,23,5.0,197000.0,985000.0
116,117,51,65,5.0,193000.0,965000.0
20,21,9,36,5.0,193000.0,965000.0
24,25,11,65,5.0,193000.0,965000.0
9,10,5,65,5.0,193000.0,965000.0
293,294,119,36,5.0,193000.0,965000.0
290,291,117,47,5.0,188000.0,940000.0
376,377,151,99,5.0,187000.0,935000.0


과제 4 결과 해석: 주문 상세 금액이 가장 큰 행은 15,500,000원(quantity 100)이고, 다음은 4,800,000원(quantity 50)이다. 두 행은 수량이 IQR 기준 정상 범위(-1~7)를 크게 벗어난 이상값 후보다. 3위부터는 985,000원 이하로, 수량 5개와 높은 단가가 겹친 일반 범위의 주문이다.

## 1. 전처리 전 상태 기록
원본 데이터의 행 수, 컬럼 수, 결측치, 중복 행 수를 확인

In [276]:
# 전처리 전 원본 데이터 상태 기록
raw_summary = []

for name, df in raw_data.items():
    raw_summary.append({
        'dataset': name,
        'rows': len(df),
        'columns': len(df.columns),
        'missing_count': df.isna().sum().sum(),
        'duplicate_rows': df.duplicated().sum()
    })

raw_summary = pd.DataFrame(raw_summary)

print('===== 1. 전처리 전 상태 =====')
display(raw_summary)

===== 1. 전처리 전 상태 =====


,dataset,rows,columns,missing_count,duplicate_rows
0,customers,152,6,6,2
1,products,102,4,0,2
2,orders,302,5,0,2
3,order_items,765,5,0,2


## 2. 문자열·타입·결측 처리
전처리된 데이터의 컬럼 타입과 결측치 현황을 확인

In [277]:
# 전처리 후 데이터 타입과 결측치 확인
print('===== 2. 문자열·타입·결측 처리 =====')

for name, df in processed_data.items():
    print(f'\n[{name}]')

    summary = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'missing_count': df.isna().sum(),
        'missing_ratio(%)': (df.isna().mean() * 100).round(2)
    })

    display(summary)

# 문자열 공백 처리 결과 확인
print('===== 문자열 처리 결과 =====')
display(customers_clean['city'].value_counts().head(10))

===== 2. 문자열·타입·결측 처리 =====

[customers]


,dtype,missing_count,missing_ratio(%)
customer_id,int64,0,0.00
name,str,0,0.00
gender,str,0,0.00
age,float64,0,0.00
city,str,0,0.00
signup_date,datetime64[us],2,1.33



[products]


,dtype,missing_count,missing_ratio(%)
product_id,int64,0,0.0
product_name,str,0,0.0
category,str,0,0.0
price,float64,0,0.0



[orders]


,dtype,missing_count,missing_ratio(%)
order_id,int64,0,0.00
customer_id,int64,0,0.00
order_date,datetime64[us],3,1.01
payment_method,str,0,0.00
order_status,str,0,0.00
order_month,str,3,1.01
order_dayofweek,str,3,1.01



[order_items]


,dtype,missing_count,missing_ratio(%)
order_item_id,int64,0,0.0
order_id,int64,0,0.0
product_id,int64,0,0.0
quantity,float64,0,0.0
unit_price,float64,0,0.0
line_total,float64,0,0.0


===== 문자열 처리 결과 =====


city
부산    23
대구    21
광주    21
서울    18
울산    17
인천    16
수원    10
성남     9
고양     8
대전     5
Name: count, dtype: int64

## 3. 중복·범주 표준화·이상값 후보
중복 개수, 주문 상태 종류, 가격·수량·단가의 이상값 후보

In [278]:
print('===== 3. 중복·범주 표준화·이상값 후보 =====')

# 전체 행 중복 확인
for name, df in processed_data.items():
    print(f'{name} 전체 행 중복: {df.duplicated().sum()}')

# 주문 상태 표준화 결과
print('\n[주문 상태 분포]')
display(orders_clean['order_status'].value_counts())

# 이상값 후보 확인
outlier_summary = pd.DataFrame({
    'column': ['age', 'price', 'quantity', 'unit_price'],
    'invalid_count': [
        ((customers_clean['age'] < 18) |
         (customers_clean['age'] > 100)).sum(),

        (products_clean['price'] <= 0).sum(),

        (order_items_clean['quantity'] <= 0).sum(),

        (order_items_clean['unit_price'] <= 0).sum()
    ]
})

print('\n[이상값 후보]')
display(outlier_summary)

===== 3. 중복·범주 표준화·이상값 후보 =====
customers 전체 행 중복: 0
products 전체 행 중복: 0
orders 전체 행 중복: 0
order_items 전체 행 중복: 0

[주문 상태 분포]


order_status
completed    179
cancelled     62
refunded      57
Name: count, dtype: int64


[이상값 후보]


,column,invalid_count
0,age,1
1,price,0
2,quantity,0
3,unit_price,0


## 4. PK/FK와 파생 컬럼 재검증
기본키 중복, 외래키 참조 관계, line_total 계산이 올바른지 확인

In [279]:
print('===== 4. PK/FK와 파생 컬럼 검증 =====')

# PK 후보 컬럼별 중복 및 결측 확인
key_columns = {
    'customers': 'customer_id',
    'products': 'product_id',
    'orders': 'order_id',
    'order_items': 'order_item_id'
}

pk_results = []

for name, key in key_columns.items():
    df = processed_data[name]

    pk_results.append({
        'dataset': name,
        'key': key,
        'duplicate_count': df[key].duplicated().sum(),
        'missing_count': df[key].isna().sum()
    })

print('\n[PK 후보 검증]')
display(pd.DataFrame(pk_results))

# FK 참조 관계 검증 (Notebook에서 생성한 결과)
print('\n[FK 검증]')
relationship_checks_after = pd.DataFrame({
    'check': relationship_checks['check'],
    'invalid_before': relationship_checks['invalid_count'],
    'invalid_after': [
        (~orders_clean['customer_id'].isin(customers_clean['customer_id'])).sum(),
        (~order_items_clean['order_id'].isin(orders_clean['order_id'])).sum(),
        (~order_items_clean['product_id'].isin(products_clean['product_id'])).sum(),
    ],
})
display(relationship_checks_after)

# line_total 계산 검증
line_total_check = (
    order_items_clean['line_total']
    == order_items_clean['quantity'] * order_items_clean['unit_price']
)

print('\n[line_total 검증]')
print('전체 행 수:', len(line_total_check))
print('계산 일치:', line_total_check.sum())
print('계산 불일치:', (~line_total_check).sum())

===== 4. PK/FK와 파생 컬럼 검증 =====

[PK 후보 검증]


,dataset,key,duplicate_count,missing_count
0,customers,customer_id,0,0
1,products,product_id,0,0
2,orders,order_id,0,0
3,order_items,order_item_id,0,0



[FK 검증]


,check,invalid_before,invalid_after
0,orders.customer_id exists in customers.custome...,2,0
1,order_items.order_id exists in orders.order_id,2,0
2,order_items.product_id exists in products.prod...,26,0



[line_total 검증]
전체 행 수: 721
계산 일치: 721
계산 불일치: 0



## 5. 전처리 전/후 비교
행 수와 컬럼 수의 변화 확인

In [280]:
print('===== 5. 전처리 전/후 비교 =====')

# comparison: Notebook에서 생성한 전처리 전후 비교표
display(comparison)

# 데이터셋별 행 수 감소 확인
for name in raw_data:
    before = len(raw_data[name])
    after = len(processed_data[name])

    print(f'{name}: {before} → {after} (감소: {before - after}행)')

===== 5. 전처리 전/후 비교 =====


,dataset,rows_raw,columns_raw,rows_processed,columns_processed
0,customers,152,6,150,6
1,products,102,4,96,4
2,orders,302,5,298,7
3,order_items,765,5,721,6


customers: 152 → 150 (감소: 2행)
products: 102 → 96 (감소: 6행)
orders: 302 → 298 (감소: 4행)
order_items: 765 → 721 (감소: 44행)


## 6. 재현 가능한 전처리 확인
같은 원본 데이터를 전처리 함수에 두 번 입력했을 때 동일한 결과가 생성되는지 확인

In [281]:
print('===== 6. 전처리 재현성 검증 =====')

# Notebook에서 정의한 전처리 함수 사용
preprocess_functions = {
    'customers': preprocess_customers,
    'products': preprocess_products,
    'orders': preprocess_orders,
    'order_items': preprocess_order_items
}

results = []

for name, func in preprocess_functions.items():

    # 동일한 원본 데이터로 전처리 두 번 실행
    result1 = func(raw_data[name])
    result2 = func(raw_data[name])

    # equals(): 데이터프레임의 값, 인덱스, 컬럼, 타입 비교
    is_same = result1.equals(result2)

    results.append({
        'dataset': name,
        'first_run_rows': len(result1),
        'second_run_rows': len(result2),
        'reproducible': is_same
    })

display(pd.DataFrame(results))

===== 6. 전처리 재현성 검증 =====


C:\Users\apdlh\AppData\Local\Temp\ipykernel_24812\3581284410.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  string_columns = result.select_dtypes(include='object').columns # 데이터 타입이 object인 컬럼들의 이름을 가져옴 (일반적으로 문자열 컬럼이 포함)
C:\Users\apdlh\AppData\Local\Temp\ipykernel_24812\3581284410.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this wa

,dataset,first_run_rows,second_run_rows,reproducible
0,customers,150,150,True
1,products,96,96,True
2,orders,300,300,True
3,order_items,752,752,True


---
---


## 0. 제출 정보
- 이름: 김혜빈
- GitHub ID: Chloe-Hyebin-Kim
- 작성일: 2026년 10월 13일
- 최종 제출 URL: https://github.com/Chloe-Hyebin-Kim/llm-data-analysis-course/blob/main/practice/chapter05/chapter05.ipynb
 
---
 
## 1. 전처리 전 상태 기록
- 각 데이터 shape:
  - customers: (152, 6)
  - products: (102, 4)
  - orders: (302, 5)
  - order_items: (765, 5)
- 결측치:
  - customers.age: 4건
  - customers.city: 2건
  - products, orders, order_items: 0건
- 전체 중복: 4개 데이터셋 모두 각 2건
  - customers: customer_id 31, 76
  - products: product_id 30, 84
  - orders: order_id 44, 205
  - order_items: order_item_id 109, 549
- 주요 ID 중복: customer_id, product_id, order_id, order_item_id 각 2건 (모두 위의 전체 행 중복과 같은 행)
- 타입 문제 후보:
  - 숫자여야 하는데 문자열로 읽힌 컬럼: `age`, `price`, `quantity`, `unit_price`
  - 날짜여야 하는데 문자열로 읽힌 컬럼: `signup_date`, `order_date`
![전처리 전 상태](images/step01_before.png)
 
### 결과 관찰
4개 데이터셋 모두 전체 행 중복이 2건씩 있었고, 주요 ID 중복도 같은 행에서 발생했다. 결측치는 customers의 age와 city에서만 보였다. 숫자와 날짜 컬럼이 모두 문자열 타입으로 읽혔는데, 이는 해당 컬럼 안에 숫자나 날짜로 해석할 수 없는 값이 섞여 있을 가능성을 의미한다.
 
### 나의 해석과 판단
가장 먼저 처리해야 할 문제는 타입 문제라고 판단했다. 숫자 컬럼이 문자열이면 `line_total` 계산과 이상값 확인이 불가능하고, 날짜가 문자열이면 월별 분석을 할 수 없다. 또한 변환 과정에서 새로운 결측이 생길 수 있기 때문에, 결측을 채우기 전에 변환을 먼저 하고 변환 실패 건수를 따로 기록해야 한다고 판단했다.
중복은 ID만 같은 것이 아니라 모든 컬럼이 같은 완전 중복이므로, 같은 데이터가 두 번 입력된 것으로 보고 제거 대상으로 판단했다.
 
### 업무·분석적 의미
중복 주문이나 중복 주문 상세가 남아 있으면 매출과 주문 수가 실제보다 크게 집계된다. 타입 문제를 해결하지 않으면 금액 계산 자체가 불가능하거나, 문자열 비교 때문에 잘못된 정렬과 집계가 발생할 수 있다.
 
### 한계와 추가 확인 사항
결측치 개수만으로는 결측이 생긴 이유를 알 수 없다. 또한 원래 결측(빈 값)과 `unknown` 같은 의미 없는 문자열은 다른 문제이므로, 변환 후 두 가지를 구분해서 확인해야 한다.
 
---
 
## 2. 문자열·타입·결측 처리
- 적용한 문자열 정리:
  - 문자열 컬럼의 앞뒤 공백 제거 (`city`의 `' 서울 '`, `payment_method`의 `' card '` 등)
- 숫자 변환 대상과 실패 건수:
  - age: 2건 (`unknown` ×2)
  - price: 2건 (`unknown` ×2)
  - quantity: 5건 (`3개`, `2개`, `5개`, `three`, `unknown`)
  - unit_price: 2건 (`unknown`, `not-a-number`)
  - 쉼표가 포함된 값(price 4건, unit_price 5건)은 쉼표를 제거한 뒤 정상 변환했다.
- 날짜 변환 대상과 실패 건수:
  - signup_date: 2건 (`2026-13-40`, `not-a-date`)
  - order_date: 3건 (`not-a-date`, `2026-02-30`, `2026/99/99`)
- 결측 처리 기준:
  - age: 원래 결측 4건 + 변환 실패 2건 = 6건을 중앙값(41세)으로 대체
  - city: 결측 2건을 `Unknown`으로 대체
  - 날짜 변환 실패: 행을 삭제하지 않고 `NaT`로 남김
  - price, quantity, unit_price 변환 실패: 분석 대상에서 제외 (3번 참조)
![타입과 결측 처리](images/step02_cleaning.png)
 
### 결과 관찰
숫자와 날짜 컬럼에서 모두 16건의 변환 실패가 발생했다. 실패 값은 `unknown`처럼 값 자체가 없는 경우, `3개`처럼 단위가 붙은 경우, `2026-02-30`처럼 형식은 맞지만 존재하지 않는 날짜인 경우로 나뉘었다. age는 원래 결측 4건에 변환 실패 2건이 더해져 결측이 6건으로 늘어났다.
 
### 나의 해석과 판단
변환 실패를 바로 채우거나 지우지 않고, 변환 전 원본 값과 비교해 실패 값을 먼저 기록했다. 실패 값을 확인하지 않고 처리하면 `3개`처럼 복구 가능한 값과 `unknown`처럼 복구할 근거가 없는 값을 구분할 수 없기 때문이다.
age는 평균보다 극단값의 영향을 덜 받는 중앙값으로 대체했다. 다만 6명의 나이가 모두 41세로 바뀌기 때문에 연령 분포가 중앙값 근처로 몰리는 정보 손실이 생긴다. city는 임의의 도시를 넣으면 지역 분석이 왜곡되므로 `Unknown`으로 구분했다.
날짜 변환 실패 행은 삭제하지 않았다. 주문 날짜가 없어도 주문 상태와 금액 정보는 유효할 수 있기 때문이다. 대신 월별 분석에서는 이 주문들이 빠진다는 점을 기록해 두었다.
 
### 업무·분석적 의미
order_date 변환에 실패한 주문 3건 중 2건은 completed 주문이다. 따라서 completed 전체 금액과 월별 completed 금액의 합계가 일치하지 않을 수 있고, 이 차이는 날짜 오류 때문이라는 점을 다음 EDA에서 설명할 수 있어야 한다.
 
### 한계와 추가 확인 사항
`3개` → 3처럼 단위만 제거하면 복구할 수 있는 값도 이번에는 변환 실패로 처리했다. 입력 규칙을 확인할 수 있다면 복구하는 것이 정보 손실을 줄이는 방법이다. 또한 결측 age를 중앙값으로 대체한 것이 연령대 분석에 영향을 주는지 추가로 확인해야 한다.
 
---
 
## 3. 중복·범주 표준화·이상값 후보
- 제거/보류한 중복:
  - 4개 데이터셋의 완전 중복 행 각 2건 제거 (customers 152→150, products 102→100, orders 302→300, order_items 765→763)
  - `order_items.order_id` 반복은 한 주문에 여러 상품이 포함되는 정상 구조이므로 중복으로 보지 않음
- 표준화한 범주값:
  - order_status: `Complete`, `complete`, `COMPLETED`, `완료` → `completed` / `cancel`, `Cancel`, `CANCELLED`, `취소` → `cancelled` / `Refund`, `refund`, `REFUNDED`, `환불` → `refunded`
  - category: 앞뒤 공백 제거 후에도 남은 `전자 기기`, `생활 용품`, `뷰 티`의 단어 사이 공백 제거 (10종 → 7종)
  - payment_method: 앞뒤 공백 제거로 4종(`kakao_pay`, `card`, `bank_transfer`, `naver_pay`)으로 정리
- 허용값 밖 값:
  - order_status: 표준화 후 `completed`, `cancelled`, `refunded` 외의 값 없음
  - 0 이하 값: price 2건(`0`, `-20000`), quantity 3건(`0`, `0`, `-1`), unit_price 1건(`0`)
- 이상값 후보 (삭제하지 않고 후보로 표시):
  - age: 150세 1건 (customer_id 33)
  - price: 9,999,999원 1건 (product_id 99)
  - quantity: 50개, 100개 2건 (IQR 기준 정상 범위 -1 ~ 7)
  - order_date: 미래 날짜 2건 (order 256: 2027-01-15, order 287: 2028-12-31)
![중복 범주 이상값](images/step03_rules.png)
 
### 결과 관찰
order_status는 표준화 후 completed 180건, cancelled 63건, refunded 57건으로 정리되었다. category는 `strip()`만으로는 정리되지 않는 단어 사이 공백 값이 3종 있었다. 가격·수량·단가에서는 변환 실패와 0 이하 값을 합쳐 products 4행, order_items 11행이 제외 대상이었다.
 
### 나의 해석과 판단
0 이하 값과 변환 실패 값은 제외 이유가 다르다. 변환 실패는 값 자체가 깨져서 계산이 불가능한 것이고, 0이나 음수는 이벤트 상품이나 반품처럼 실제 의미가 있을 수 있는 값이다. 이번에는 정상 주문 분석을 위해 둘 다 제외했지만, 제외 목록을 따로 출력해 근거를 남겼다.
이상값 후보는 자동으로 삭제하지 않았다. quantity 100은 입력 오류일 수도 있지만 대량 주문일 수도 있다. 특히 product_id 99의 상품 가격은 9,999,999원이지만 실제 주문 상세의 판매 단가는 187,000원이었다. 따라서 이 값은 주문 금액이 아니라 상품 마스터의 가격 입력 오류일 가능성이 높고, `line_total`은 unit_price로 계산하므로 매출 계산에는 직접 영향을 주지 않는다. 이처럼 이상값은 다른 테이블과 비교해야 의미를 판단할 수 있기 때문에 삭제보다 검토가 먼저라고 판단했다.
 
### 업무·분석적 의미
category 표기가 통일되지 않으면 같은 카테고리의 매출이 여러 그룹으로 나뉘어 집계된다. quantity 50, 100인 주문 상세의 금액은 각각 4,800,000원, 15,500,000원으로 일반 주문보다 매우 크다. 이 중 quantity 50인 행은 completed 주문에 속하므로 completed 매출에 직접 영향을 준다.
 
### 한계와 추가 확인 사항
IQR 기준은 분포상 멀리 떨어진 값을 찾는 기준일 뿐, 오류 여부를 확정하지 못한다. 대량 주문 여부, 미래 날짜의 입력 경위, 150세 고객의 실제 나이는 원본 시스템이나 담당자 확인이 필요하다.
 
---
 
## 4. PK/FK와 파생 컬럼 재검증
- 없는 `customer_id`: 9991, 9992 (orders 2건)
- 없는 `order_id`: 9991, 9992 (order_items 2건)
- 없는 `product_id`: 26건
  - 원본 products에 없는 상품: 2건 (9991, 9992)
  - 가격 필터로 제외한 상품(18, 48, 73, 95)을 참조: 24건
- 생성한 파생 컬럼:
  - `line_total = quantity × unit_price`
  - `order_month`, `order_dayofweek`
- `line_total` 검증: 721행 중 721행 일치, 불일치 0건
![관계와 파생 컬럼 검증](images/step04_validation.png)
 
### 결과 관찰
원본 상태에서 product_id 불일치는 2건이었지만, 가격이 0 이하이거나 변환에 실패한 상품 4개를 제외한 뒤에는 26건으로 늘어났다. 관계가 깨진 행을 분석 대상에서 제외한 결과 orders 2행, order_items 31행이 빠졌고, 제외 후 불일치는 모두 0건이 되었다. order_items 31행에는 고객이 없는 주문 2건에 딸린 주문 상세 3행도 포함된다.
 
### 나의 해석과 판단
전처리에서 한 테이블의 행을 지우면 그 행을 참조하던 다른 테이블의 관계가 깨진다. 이번 결과에서 product 불일치가 2건에서 26건으로 늘어난 것이 그 예이다. 따라서 PK/FK 검증은 원본에서 한 번만 하는 것이 아니라 전처리 후에 반드시 다시 해야 한다고 판단했다.
관계가 깨진 주문 상세는 상품 카테고리나 고객 정보를 연결할 수 없어 카테고리별·고객별 분석이 불가능하므로 분석 대상에서 제외했다. 다만 이 선택 때문에 상품 4개의 정상 주문 기록까지 함께 빠졌다는 정보 손실이 있다.
 
---
 
## 5. 전처리 전/후 비교
| 항목 | 처리 전 | 처리 후 | 변화 이유 |
| --- | ---: | ---: | --- |
| 행 수 | customers 152 / products 102 / orders 302 / order_items 765 | customers 150 / products 96 / orders 298 / order_items 721 | 완전 중복 제거, 가격·수량·단가 변환 실패와 0 이하 제외, FK 불일치 제외 |
| 결측 | 6건 (age 4, city 2) | 11건 (signup_date 2, order_date 3 + 파생 컬럼 order_month·order_dayofweek 각 3) | age·city는 대체, 날짜 변환 실패는 NaT로 남김 |
| 중복 | 각 2건, 총 8건 | 0건 | 완전 중복 행 제거 |
| 변환 실패 | 16건 (숫자 11, 날짜 5) | 날짜 5건만 NaT로 남음 | age는 중앙값 대체, 가격·수량·단가 실패 행은 제외 |
 
![전처리 전후 비교](images/step05_before_after.png)
 
### 결과 관찰
모든 데이터셋에서 행 수가 줄었고, 특히 order_items는 765행에서 721행으로 44행이 줄었다(중복 2, 값 오류 11, FK 불일치 31). 결측은 오히려 6건에서 11건으로 늘었는데, 이는 날짜 변환 실패를 `NaT`로 남겼기 때문이다.
 
### 나의 해석과 판단
행 수 감소는 모두 기록한 규칙에 따른 것이므로 의도한 결과라고 판단했다. 근거는 단계별 감소 건수의 합이 전체 감소 건수와 일치한다는 점이다. 결측이 늘어난 것도 날짜 오류 행을 삭제하지 않고 남기기로 한 선택의 결과이다. 결측 개수만 보면 전처리가 나빠진 것처럼 보일 수 있으므로, 결측이 늘어난 이유를 함께 기록해야 한다.
 
### 업무·분석적 의미
order_items가 약 5.8% 줄어든 만큼 매출 합계도 원본보다 작아진다. 따라서 이후 보고서의 매출이 원본 시스템의 매출과 다르다면 이 전처리 규칙 때문이라는 점을 설명할 수 있어야 한다.
 
### 한계와 추가 확인 사항
FK 불일치 제외는 가격 오류가 있는 상품 4개의 주문 기록까지 함께 제거했다. 상품 가격 오류만 있고 주문 자체는 정상이었다면 매출이 과소 집계되었을 수 있다. 상품 가격을 복구할 수 있는지 원본 확인이 필요하다.
 
---
 
## 6. 재현 가능한 전처리 확인
- 생성된 clean CSV:
  - data/processed/customers_clean.csv
  - data/processed/products_clean.csv
  - data/processed/orders_clean.csv
  - data/processed/order_items_clean.csv
- 생성된 요약 보고서:
  - reports/ch05_preprocessing_summary.md
- `python scripts/preprocess_data.py` 재실행 결과:
  - 프로젝트 루트에서 `python -m scripts.preprocess_data`로 실행했다. 입력 Raw 경로는 `practice/chapter05/data/raw`였고, 실행 후 `data/processed/`에 4개 clean CSV와 `reports/ch05_preprocessing_summary.md`가 생성되었다.
- 재실행 후 달라진 점:
  - 스크립트 결과 행 수는 customers/products/orders/order_items 순서로 150/96/300/752행이었다. Notebook 결과인 150/96/298/721행과 비교하면 customers와 products는 같고, orders는 2행, order_items는 31행 더 많았다.
![재실행 결과](images/step06_reproducible.png)
 
### 나의 해석과 판단
처음에는 `python scripts/preprocess_data.py`를 실행했을 때 `ModuleNotFoundError`가 발생했다. 파일 경로로 스크립트를 실행하면 Python이 `scripts` 폴더를 기준으로 모듈을 찾기 때문에 프로젝트 루트의 `src` 패키지를 찾지 못한 것이다. 프로젝트 루트에서 모듈 방식(`python -m scripts.preprocess_data`)으로 실행해 이 문제를 해결했다.
스크립트 결과와 Notebook 결과의 행 수가 다른 이유는 두 곳의 처리 규칙이 완전히 같지 않기 때문이다. 현재 `src/preprocessing.py`의 스크립트 처리에는 Notebook에서 수행한 category 표준화와 FK 불일치 행 제외가 반영되어 있지 않다. 스크립트는 FK 불일치를 검사해 보고서에 기록하지만 해당 행을 제거하지 않는다. 반면 Notebook은 FK 불일치 주문 2건과 이에 연결되거나 유효하지 않은 주문 상세 31건을 제외해 orders는 298행, order_items는 721행이 되었다. category 표준화는 행 수 차이의 원인은 아니지만, 값의 표기 결과가 달라질 수 있다. 따라서 현재 스크립트는 실행을 반복할 수는 있어도 Notebook과 동일한 처리 결과를 재현하지는 않는다.
Notebook은 셀 실행 순서와 이전에 만든 변수에 영향을 받기 때문에, 한 번 성공한 결과가 다시 실행했을 때도 같다는 보장이 없다. 이번에 새로 추가한 처리 규칙도 Notebook 안에만 있으면 다른 사람이나 다른 환경에서 같은 결과를 만들 수 없다. 재현 가능한 전처리를 위해서는 처리 규칙을 함수나 모듈로 옮기고, 스크립트 실행 결과와 Notebook 결과를 비교해야 한다.
 
Notebook의 전처리 함수(preprocess_*)도 FK 불일치 제외를 포함하지 않아, 두 번 실행한 결과는 서로 같지만 Notebook 본문의 최종 결과와는 다르다.
---
 
## 7. Chapter 05 최종 판단
### 내가 정한 전처리 원칙 3가지
1. 변환하거나 지우기 전에 먼저 기록한다.
   → 변환 실패 값, 제외되는 행, FK 불일치 원인을 처리 전에 출력해 두어야 처리 기준이 맞았는지 나중에 검증할 수 있다.
2. 제외 이유를 구분한다.
   → 변환 실패, 0 이하 값, 이상값 후보, FK 불일치는 서로 다른 문제이므로 하나의 필터로 한꺼번에 지우지 않는다.
3. 행을 지운 뒤에는 관계를 다시 검증한다.
   → 한 테이블에서 지운 행 때문에 다른 테이블의 관계가 깨질 수 있다. 이번 실습에서 product 불일치가 2건에서 26건으로 늘어났다.
### 가장 위험하다고 생각한 자동 처리 1가지와 이유
가장 위험한 자동 처리는 `errors='coerce'` 변환 후 `> 0` 필터를 바로 적용하는 것이다. 변환에 실패한 값은 NaN이 되고, NaN은 `> 0` 조건에서 False가 되어 아무 기록 없이 사라진다. 이번 데이터에서도 order_items 11행이 제외되었는데, 그중 7행은 0 이하가 아니라 변환 실패였다. 별도로 기록하지 않았다면 `3개`처럼 복구할 수 있는 값까지 이유도 모른 채 잃었을 것이다.
 
### 다음 EDA에서 특히 주의할 데이터 특성
- 금액 분석은 completed 주문만 사용하되, 날짜 변환에 실패한 completed 주문 2건 때문에 completed 전체 합계와 월별 합계가 다를 수 있다.
- 미래 날짜 주문(2027-01-15 completed, 2028-12-31 cancelled)이 월별 분석에 별도 월로 나타날 수 있다.
- quantity 50, 100인 주문 상세가 카테고리별·고객별 매출을 크게 끌어올릴 수 있으므로, 이상값 후보를 포함한 결과와 제외한 결과를 비교해야 한다.
- category는 표준화된 7종을 기준으로 집계해야 한다.
### 현재 전처리의 한계
- 이상값 후보(150세, 9,999,999원, 수량 50·100, 미래 날짜)는 원본 확인 없이 후보로만 표시했다.
- 단위가 붙은 수량(`3개`)처럼 복구 가능한 값도 제외했다.
- 가격 오류 상품 4개를 제외하면서 해당 상품의 주문 기록도 함께 빠졌다.
- Notebook에서 추가한 처리 규칙이 스크립트와 소스 모듈에는 반영되지 않았다.

## 최종 제출 체크
- [O] raw 원본을 덮어쓰지 않았습니다.
- [O] 처리 전/후 비교를 남겼습니다.
- [O] 처리 기준과 이유를 자신의 말로 작성했습니다.
- [O] clean CSV와 재실행 결과를 확인했습니다.
- [O] 개인정보/Secret이 없습니다.
- [O] `chapter05/chapter05.ipynb`가 GitHub에서 정상 표시됩니다.
- [O] 최종 Notebook 파일 URL을 제출합니다.

## 25. 정리

이번 장에서는 다음 내용을 실습했습니다.

- 원본 데이터와 전처리 데이터 분리
- 결측치 개수와 비율 확인
- 나이 결측치 중앙값 대체, 도시 결측치 Unknown 처리
- 중복 행과 주요 ID 중복 점검
- 문자열 앞뒤 공백 제거와 상태값 표기 통일
- 날짜형 변환과 주문 월/요일 파생 컬럼 생성
- 숫자형 변환과 변환 실패 건수 확인
- 이상값 후보 확인과 실습용 처리 기준 적용
- 주문 상세 금액 `line_total` 생성
- 파일 간 키 관계 재확인
- 전처리 결과 CSV와 요약 보고서 저장
- 전처리 함수를 `src/preprocessing.py` 소스 모듈로 분리

다음 장에서는 전처리된 데이터를 바탕으로 탐색적 데이터 분석, 즉 EDA를 수행합니다.
